# 03. Phân tích sai số Ridge và cải thiện mô hình

Ở notebook 02, Ridge có RMSE log trung bình thấp hơn DummyRegressor
trên cùng 5 fold. Điều này cho thấy việc sử dụng các đặc trưng căn nhà
đã giúp cải thiện dự đoán so với mốc đơn giản.

Tuy nhiên, sai số trung bình chưa cho biết mô hình dự đoán sai
những căn nào hoặc có nhóm căn nào thường sai nhiều hơn.

Notebook này tìm hiểu:

1. Những căn nào có sai số dự đoán lớn nhất?
2. Các căn dự đoán sai nhiều có đặc điểm chung nào đáng kiểm tra?
3. Những căn đó có liên quan đến sai số cao của fold 3 không?

Phân tích sử dụng dự đoán out-of-fold: mỗi căn được dự đoán bởi
mô hình không được huấn luyện trên chính căn đó. Các kết quả sẽ
được dùng để đề xuất hướng thử nghiệm, chưa đủ để kết luận
nguyên nhân gây sai số.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Tìm thư mục dự án từ thư mục làm việc hiện tại.
current_dir = Path.cwd()

PROJECT_DIR = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "data" / "raw").is_dir()),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/raw.")

train = pd.read_csv(PROJECT_DIR / "data" / "raw" / "train.csv")

X = train.drop(columns=["Id", "SalePrice"]).copy()
y = train["SalePrice"].copy()

# Giống notebook 02: mã loại nhà là biến phân loại.
X["MSSubClass"] = X["MSSubClass"].astype("string")

y_log = np.log(y)

print("X:", X.shape)
print("y:", y.shape)

X: (1460, 79)
y: (1460,)


## 2. Áp dụng lại mô hình của Ridge từ 02

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0, solver="lsqr")),
])

### 2.1 Tạo dự đoán out-of-fold và lưu số fold

Ta dùng .iloc để chọn dòng theo vị trí, còn oof_log_predictions[val_idx] = ... đặt dự đoán trở lại đúng các vị trí ấy

In [7]:
from sklearn.base import clone
from sklearn.model_selection import KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Mỗi vị trí tương ứng với một dòng trong X.
oof_log_predictions = np.full(len(X), np.nan)
fold_numbers = np.zeros(len(X), dtype=int)

for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):
    # Tạo pipeline mới cho mỗi fold.
    model = clone(ridge_pipeline)

    # Chỉ học trên 4 phần huấn luyện.
    model.fit(X.iloc[train_idx], y_log.iloc[train_idx])

    # Dự đoán phần còn lại, rồi đặt kết quả vào đúng vị trí dòng.
    oof_log_predictions[val_idx] = model.predict(X.iloc[val_idx])
    fold_numbers[val_idx] = fold

print("Số dự đoán còn thiếu:", np.isnan(oof_log_predictions).sum())
print("Số căn trong từng fold:")
print(pd.Series(fold_numbers).value_counts().sort_index())

Số dự đoán còn thiếu: 0
Số căn trong từng fold:
1    292
2    292
3    292
4    292
5    292
Name: count, dtype: int64


In [8]:
results = pd.DataFrame({
    "Id": train["Id"].to_numpy(),
    "fold": fold_numbers,
    "actual_price": y.to_numpy(),
    "predicted_log": oof_log_predictions,
})

# Đưa dự đoán về đơn vị USD.
results["predicted_price"] = np.exp(results["predicted_log"])

# Giữ cả hướng và độ lớn sai số.
results["error_log"] = (
    results["predicted_log"] - np.log(results["actual_price"])
)
results["abs_error_log"] = results["error_log"].abs()

# Sai số USD: dương là dự đoán cao, âm là dự đoán thấp.
results["error_usd"] = (
    results["predicted_price"] - results["actual_price"]
)

display(results.head())

,Id,fold,actual_price,predicted_log,predicted_price,error_log,abs_error_log,error_usd
0,1,3,208500,12.231039,205056.141860,-0.016655,0.016655,-3443.858140
1,2,5,181500,12.208418,200469.667432,0.099407,0.099407,18969.667432
2,3,3,223500,12.249877,208955.594263,-0.067290,0.067290,-14544.405737
3,4,2,140000,12.117146,182982.589814,0.267749,0.267749,42982.589814
4,5,4,250000,12.592918,294465.854748,0.163702,0.163702,44465.854748


In [9]:
fold_rmse = (
    results.groupby("fold")["error_log"]
    .apply(lambda errors: np.sqrt(np.mean(errors ** 2)))
    .rename("rmse_log")
)

display(fold_rmse.round(5))

fold
1    0.13397
2    0.12702
3    0.22841
4    0.13340
5    0.11693
Name: rmse_log, dtype: float64

$\rightarrow$ Kết quả các fold đúng như bản 2.0

## 3. Tìm các căn có sai số log lớn nhất

In [10]:
largest_errors = results.sort_values(
    "abs_error_log",
    ascending=False
)

display(largest_errors.head(10).round(4))

,Id,fold,actual_price,predicted_log,predicted_price,error_log,abs_error_log,error_usd
1298,1299,3,160000,14.9247,3.031858e+06,2.9418,2.9418,2.871858e+06
523,524,3,184750,13.8030,9.875692e+05,1.6762,1.6762,8.028192e+05
632,633,5,82500,12.1071,1.811522e+05,0.7865,0.7865,9.865220e+04
88,89,2,85000,10.6388,4.172424e+04,-0.7116,0.7116,-4.327576e+04
825,826,4,385000,12.1627,1.915055e+05,-0.6983,0.6983,-1.934945e+05
1423,1424,5,274970,11.8363,1.381772e+05,-0.6881,0.6881,-1.367928e+05
462,463,2,62383,11.7273,1.239101e+05,0.6863,0.6863,6.152707e+04
968,969,4,37900,11.2217,7.473317e+04,0.6790,0.6790,3.683317e+04
1324,1325,3,147000,12.5520,2.826688e+05,0.6538,0.6538,1.356688e+05
30,31,1,40000,11.2430,7.634261e+04,0.6464,0.6464,3.634261e+04


In [16]:
fold_3_errors = largest_errors[
    largest_errors["fold"] == 3
]

display(fold_3_errors.head(10).round(4))

,Id,fold,actual_price,predicted_log,predicted_price,error_log,abs_error_log,error_usd
1298,1299,3,160000,14.9247,3.031858e+06,2.9418,2.9418,2.871858e+06
523,524,3,184750,13.8030,9.875692e+05,1.6762,1.6762,8.028192e+05
1324,1325,3,147000,12.5520,2.826688e+05,0.6538,0.6538,1.356688e+05
970,971,3,135000,11.3103,8.165733e+04,-0.5027,0.5027,-5.334267e+04
533,534,3,39300,11.0616,6.367856e+04,0.4826,0.4826,2.437856e+04
1270,1271,3,260000,12.0245,1.667954e+05,-0.4439,0.4439,-9.320460e+04
1453,1454,3,84500,11.7626,1.283620e+05,0.4181,0.4181,4.386203e+04
1443,1444,3,121000,11.3581,8.565782e+04,-0.3454,0.3454,-3.534218e+04
142,143,3,166000,11.6966,1.201671e+05,-0.3231,0.3231,-4.583289e+04
557,558,3,108000,11.2716,7.855751e+04,-0.3183,0.3183,-2.944249e+04


**Nhận xét:** 
Giá trị bất thường có log sai số cao nhất đúng như EDA đã phân tích là id 524 và 1299, nó đang góp phần đáng kể vào sai số của fold = 3. Và error_log của hai điểm số vẫn dương tức là mô hình Ridge đang đoán cao hơn so với thực tế.

## 4. Kiểm tra mức độ ảnh hưởng của hai căn nhà này lên fold 3

In [19]:
fold_3 = results[results["fold"] == 3].copy()

fold_3["squared_error_log"] = fold_3["error_log"] ** 2

total_squared_error = fold_3["squared_error_log"].sum()

two_houses = fold_3[fold_3["Id"].isin([524, 1299])]

contribution = (
    two_houses["squared_error_log"].sum()
    / total_squared_error
    * 100
)

display(
    two_houses[
        ["Id", "actual_price", "predicted_price", "squared_error_log"]
    ].round(4)
)

print(
    "Hai căn chiếm "
    f"{contribution:.2f}% tổng bình phương sai số của fold 3."
)

,Id,actual_price,predicted_price,squared_error_log
523,524,184750,9.875692e+05,2.8098
1298,1299,160000,3.031858e+06,8.6539


Hai căn chiếm 75.25% tổng bình phương sai số của fold 3.


In [20]:
remaining = fold_3[~fold_3["Id"].isin([524, 1299])]

rmse_all = np.sqrt(np.mean(fold_3["squared_error_log"]))
rmse_remaining = np.sqrt(
    np.mean(remaining["squared_error_log"])
)

print(f"RMSE cả 292 căn:       {rmse_all:.5f}")
print(f"RMSE 290 căn còn lại: {rmse_remaining:.5f}")
print(f"Số căn còn lại:       {len(remaining)}")

RMSE cả 292 căn:       0.22841
RMSE 290 căn còn lại: 0.11402
Số căn còn lại:       290


**Nhận xét**: Hai căn nhà này ảnh hưởng rất đáng kể đến điểm fold = 3


In [22]:
# Ghép thông tin căn nhà với dự đoán out-of-fold theo Id.
comparison = train.merge(
    results[
        ["Id", "fold", "predicted_price", "error_log", "abs_error_log"]
    ],
    on="Id",
    validate="one_to_one"
)

# Xem nhóm căn cùng khu vực và mức chất lượng.
comparison = comparison[
    (comparison["Neighborhood"] == "Edwards")
    & (comparison["OverallQual"].between(8, 10))
]

columns = [
    "Id", "fold", "GrLivArea", "OverallQual",
    "SaleType", "SaleCondition", "SalePrice",
    "predicted_price", "error_log"
]

display(
    comparison[columns]
    .sort_values("GrLivArea")
    .round(3)
)

,Id,fold,GrLivArea,OverallQual,SaleType,SaleCondition,SalePrice,predicted_price,error_log
724,725,3,1698,9,WD,Normal,320000,291675.180,-0.093
523,524,3,4676,10,New,Partial,184750,987569.162,1.676
1298,1299,3,5642,10,New,Partial,160000,3031857.818,2.942


**Nhận xét**: Kết quả cho thấy 3 căn cùng thuộc fold 3, nhưng hai căn 524 và 1299 có diện tích lớn hơn rất nhiều và giá thực tế lại thấp hơn căn 725.

Id 524: 4.676 ft², giá 184.750 USD; Ridge dự đoán khoảng 987.569 USD.

Id 1299: 5.642 ft², giá 160.000 USD; Ridge dự đoán khoảng 3.031.858 USD.

Id 725: 1.698 ft², giá 320.000 USD; Ridge dự đoán khoảng 291.675 USD.

In [23]:
# Tái tạo chính xác train/validation của fold 3.
for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):
    if fold == 3:
        fold_3_train = train.iloc[train_idx]
        break

print("Số căn huấn luyện fold 3:", len(fold_3_train))
print("Diện tích lớn nhất:", fold_3_train["GrLivArea"].max())

display(
    fold_3_train[
        ["Id", "GrLivArea", "OverallQual", "SalePrice"]
    ]
    .sort_values("GrLivArea", ascending=False)
    .head(10)
)

Số căn huấn luyện fold 3: 1168
Diện tích lớn nhất: 4476


,Id,GrLivArea,OverallQual,SalePrice
1182,1183,4476,10,745000
691,692,4316,10,755000
185,186,3608,10,475000
304,305,3493,7,295000
1268,1269,3447,8,381000
635,636,3395,6,200000
769,770,3279,8,538000
1353,1354,3238,8,410000
496,497,3228,8,430000
608,609,3194,8,359100


## 5. Cải thiện mô hình Ridge

In [24]:
from sklearn.model_selection import GridSearchCV

alpha_values = [0.01, 0.1, 1.0, 10.0, 100.0]

search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={"model__alpha": alpha_values},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    return_train_score=True,
    error_score="raise",
)

search.fit(X, y_log)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...ver='lsqr'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitt

### Dùng GridCV để so sánh và tìm ra giá trị $\alpha$ tối ưu

In [25]:
from sklearn.model_selection import GridSearchCV

alpha_values = [0.01, 0.1, 1.0, 10.0, 100.0]

search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={"model__alpha": alpha_values},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    return_train_score=True,
    error_score="raise",
)

search.fit(X, y_log)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...ver='lsqr'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitt

In [26]:
tuning_results = pd.DataFrame({
    "alpha": search.cv_results_["param_model__alpha"],
    "rmse_log_mean": -search.cv_results_["mean_test_score"],
    "rmse_log_std": search.cv_results_["std_test_score"],
    "rank": search.cv_results_["rank_test_score"],
})

display(tuning_results.sort_values("rank").round(5))

print("Alpha tốt nhất:", search.best_params_["model__alpha"])
print("RMSE log CV tốt nhất:", round(-search.best_score_, 5))

,alpha,rmse_log_mean,rmse_log_std,rank
3,10.00,0.14577,0.03956,1
4,100.00,0.14621,0.04016,2
2,1.00,0.14795,0.04070,3
1,0.10,0.15064,0.04451,4
0,0.01,0.15236,0.04476,5


Alpha tốt nhất: 10.0
RMSE log CV tốt nhất: 0.14577


**Nhận xét**: Giá trị alpha = 10.0 là tối ưu nhất

### Ta thử việc cải thiện này có ảnh hướng đến các fold như thế nào?

In [27]:
fold_comparison = pd.DataFrame({
    "fold": range(1, 6),
    "alpha_1": [
        -search.cv_results_[f"split{i}_test_score"][
            alpha_values.index(1.0)
        ]
        for i in range(5)
    ],
    "alpha_10": [
        -search.cv_results_[f"split{i}_test_score"][
            alpha_values.index(10.0)
        ]
        for i in range(5)
    ],
})

fold_comparison["improvement"] = (
    fold_comparison["alpha_1"] - fold_comparison["alpha_10"]
)

display(fold_comparison.round(5))

,fold,alpha_1,alpha_10,improvement
0,1,0.13397,0.13746,-0.00349
1,2,0.12702,0.12759,-0.00057
2,3,0.22841,0.22347,0.00494
3,4,0.13340,0.12644,0.00696
4,5,0.11693,0.11390,0.00302


**Nhận xét**: Tăng mức phạt giúp cải thiện nhẹ kết quả trung bình nhưng độ cải thiên không đáng kể, chưa giải quyết rõ rệt sai số cao ở fold 3.

# Kết luận thí nghiệm điều chỉnh alpha

Trong các giá trị alpha đã thử, Ridge với `alpha = 10` đạt RMSE
trên log giá trung bình **0,14577**, so với **0,14794** của
baseline `alpha = 1`. Mức giảm khoảng **1,47%**.

Kết quả cải thiện ở fold 3, 4 và 5, nhưng sai số tăng nhẹ ở fold 1
và 2. Fold 3 vẫn có sai số cao hơn các fold còn lại.

Như vậy, điều chỉnh alpha trong phạm vi khảo sát chỉ cải thiện
nhẹ kết quả trung bình và chưa giải quyết rõ rệt sai số ở fold 3.
Giá trị `alpha = 10` là tốt nhất trong các giá trị đã thử, chưa
thể xem là tối ưu trên mọi giá trị hoặc trên dữ liệu mới.

**Hướng tiếp theo:** thử một mô hình khác trên cùng dữ liệu và
5 fold để so sánh với Ridge.